# Pseudonymizer - fine-tune on Colab

Before running: **Runtime -> Change runtime type -> A100 GPU** (Colab Pro). Enable *Background execution* for long runs.

Data (`train.parquet`, `dev.parquet`, `dev.jsonl`, optionally `dev_judges.*`) must be in Google Drive at `MyDrive/ai-camp/` (or `MyDrive/ai-camp/silver/`). Code is cloned from GitHub; every run is copied back to `MyDrive/ai-camp/runs/<name>/`.

The last cell scores the run on the organizers' test set and shows the test leaderboard over every run on Drive with the history of runs (`MyDrive/ai-camp/runs/leaderboard_all.md`).

Timing on A100, 24k docs x 3 epochs: base models ~25 min, large ~1-1.5 h.

In [ ]:
#@title 1. Code
REPO = "EnderLancer/anonimimizer"  #@param {type:"string"}
BRANCH = "dmelashych/datagen"  #@param {type:"string"}
import os
# private repo: add a GitHub token as Colab secret GH_TOKEN (left sidebar -> key icon -> enable notebook access)
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists('/content/ai-camp/train'):
    !rm -rf /content/ai-camp && git clone -q --branch "$BRANCH" "$url" /content/ai-camp
else:
    !cd /content/ai-camp && git pull -q
assert os.path.exists('/content/ai-camp/train'), 'clone failed: check REPO/BRANCH and the GH_TOKEN secret'
%cd /content/ai-camp/train
!pip install -q -e . 2>&1 | tail -1
import os; os.environ.update(TRANSFORMERS_VERBOSITY='error', PYTHONWARNINGS='ignore', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('GPU:', gpu)
if 'A100' not in gpu and 'L4' not in gpu:
    print('WARNING: not an A100 - Runtime -> Change runtime type -> A100 GPU (Colab Pro); on T4 halve every batch size')

In [ ]:
#@title 2. Data from Google Drive
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil
from tqdm.auto import tqdm
DRIVE = '/content/drive/MyDrive/ai-camp'
os.makedirs('/content/ai-camp/data/silver', exist_ok=True); os.makedirs(f'{DRIVE}/runs', exist_ok=True)
src = f'{DRIVE}/silver' if os.path.exists(f'{DRIVE}/silver/train.parquet') else DRIVE
files = glob.glob(f'{src}/*.parquet') + glob.glob(f'{src}/*.jsonl')
assert files, f'no parquet/jsonl files found in {DRIVE} or {DRIVE}/silver'
for f in tqdm(files, desc='copy from Drive'):
    shutil.copy(f, '/content/ai-camp/data/silver/')
for need in ('train.parquet', 'dev.parquet', 'dev.jsonl'):
    assert os.path.exists(f'/content/ai-camp/data/silver/{need}'), f'{need} is missing in {src} - upload it to Drive first'
!ls -la /content/ai-camp/data/silver/
assert os.path.exists('/content/ai-camp/data/evaluate.py'), 'data/evaluate.py missing in the repo'
print('OK, data from', src)

In [ ]:
#@title 3. Hugging Face - every trained model is pushed to a PRIVATE repo <HF_USER>/pseudo-<run>
HF_USER = ""  #@param {type:"string"}  # your HF username or org; empty = do not push
if HF_USER:
    import os
    from google.colab import userdata
    from huggingface_hub import login
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # Colab: left sidebar -> Secrets -> HF_TOKEN (write access), enable notebook access
    login(os.environ['HF_TOKEN'])
else:
    print('HF_USER is empty: models are NOT pushed to the Hub')

In [ ]:
#@title 4. Train one model
MODEL = "benjamin/roberta-base-wechsel-ukrainian"  #@param ["benjamin/roberta-base-wechsel-ukrainian", "FacebookAI/xlm-roberta-base", "microsoft/mdeberta-v3-base", "benjamin/roberta-large-wechsel-ukrainian", "FacebookAI/xlm-roberta-large", "Goader/liberta-large-v2"] {allow-input: true}
RUN = "wechsel_base_full"  #@param {type:"string"}
EPOCHS = 3  #@param {type:"number"}
LR = 4e-5  #@param {type:"number"}  # base: 4e-5 at batch 64; large: 2e-5
BATCH = 64  #@param {type:"integer"}  # A100 40GB: base 64, large 32; T4 16GB: base 16, large 8 + GRAD_ACCUM 2
GRAD_ACCUM = 1  #@param {type:"integer"}
N_DOCS = 0  #@param {type:"integer"}
DEV_DOCS = 500  #@param {type:"integer"}
extra = (f' --n-docs {N_DOCS}' if N_DOCS else '') + (f' --push-to {HF_USER}/pseudo-{RUN}' if HF_USER else '')
# --skip-if-done: if the same model + hyper-parameters + data version is already on Drive, nothing is retrained
!python scripts/train.py --model $MODEL --out runs/$RUN --epochs $EPOCHS --lr $LR --batch $BATCH --grad-accum $GRAD_ACCUM --dev-docs $DEV_DOCS --skip-if-done $DRIVE/runs/$RUN $extra
!rm -rf runs/$RUN/checkpoints && mkdir -p $DRIVE/runs/$RUN && cp -r runs/$RUN/. $DRIVE/runs/$RUN/ && echo saved to $DRIVE/runs/$RUN - model, config.json with data version and fingerprint, train.log

In [ ]:
#@title 5. (optional) CPU timing estimate - the organizers' limit is 2 h for 240 docs on CPU
!head -n 30 ../data/test/court_cases.jsonl > /tmp/test30.jsonl
import time; t0 = time.time()
!python scripts/predict.py --model runs/$RUN/model --input /tmp/test30.jsonl --output /tmp/cpu_sub.jsonl --device cpu --threads 4 --batch 8
dt = (time.time() - t0) / 30
import json, os
p = f'{DRIVE}/runs/cpu_timing_train.json'; timing = json.load(open(p)) if os.path.exists(p) else {}
timing[RUN] = dt; json.dump(timing, open(p, 'w'))  # the leaderboard reads runs/cpu_timing_*.json
print(f'{dt:.2f} s/doc on Colab CPU -> test (240 docs) ~ {dt*240/60:.1f} min  (a laptop CPU is usually 1.5-2x faster)')

In [ ]:
#@title 6. Leaderboard on the organizers' test set (this run + every run on Drive) + history of runs -> Drive
from IPython.display import Markdown, display
!python scripts/leaderboard.py --drive $DRIVE --name leaderboard_all --all --runs $RUN
display(Markdown(open(f'{DRIVE}/runs/leaderboard_all.md', encoding='utf-8').read()))